# Técnicas de preprocesamiento usando PyTorch
Se usará el dataset de California Housing para este ejercicio.

In [1]:
from sklearn.datasets import fetch_california_housing
x, y = fetch_california_housing(return_X_y=True, as_frame=True)
x.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


# Partición

In [2]:
from sklearn.model_selection import train_test_split

# 80 / 10 / 10  —  primero separamos test (10%), luego val del resto
x_train, x_temp, y_train, y_temp = train_test_split(x, y, test_size=0.20, random_state=42)
x_val,   x_test,  y_val,   y_test  = train_test_split(x_temp, y_temp, test_size=0.50, random_state=42)

print(f"Train : {len(x_train):>5}  ({len(x_train)/len(x)*100:.0f}%)")
print(f"Val   : {len(x_val):>5}  ({len(x_val)/len(x)*100:.0f}%)")
print(f"Test  : {len(x_test):>5}  ({len(x_test)/len(x)*100:.0f}%)")

Train : 16512  (80%)
Val   :  2064  (10%)
Test  :  2064  (10%)


In [ ]:
import torch
import torch.nn as nn

# Columnas que se procesan (todas excepto Latitude y Longitude)
COLS_TO_PROCESS = [c for c in x.columns if c not in ("Latitude", "Longitude")]
COLS_PROCESS_IDX = [list(x.columns).index(c) for c in COLS_TO_PROCESS]

class Preprocessor(nn.Module):
    def __init__(self, x_train_tensor: torch.Tensor, cols_idx: list):
        super().__init__()
        self.cols_idx = cols_idx

        data = x_train_tensor[:, cols_idx]

        # En este caso los calculo dentro de la clase
        mean = data.mean(dim=0)
        std  = data.std(dim=0)

        self.register_buffer("mean",  mean)
        self.register_buffer("std",   std)
        self.register_buffer("lower", mean - 3 * std)
        self.register_buffer("upper", mean + 3 * std)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.clone()
        # Clipping para quitar los outliers
        x[:, self.cols_idx] = torch.clamp(x[:, self.cols_idx], min=self.lower, max=self.upper)
        
        # Standard scaling
        x[:, self.cols_idx] = (x[:, self.cols_idx] - self.mean) / (self.std + 1e-8)
        return x